Inception Score

$D_{KL} = p(y\vert x) \left( \log(p(y\vert x)) - \log(p(y)) \right)$

$IS = \exp(D_{KL})$

There are three main terms in the above formula:

1.   Conditional Probability Distribution — $p(y\vert x)$.
2.   Marginal Probability Distribution — $p(y)$
3.   KL-Divergence — KL-Divergence between conditional probability $p(y\vert x)$ and marginal probability $p(y)$

How to calculate the Inception Score?

     

1.   Pass the generated images through the Inception model to get the conditional label distribution $p(y|x)$
2. Calculate the marginal probability distribution $p(y)$
3. Calculate the KL Divergence between $p(y)$ and $p(y|x)$
4. Calculate the sum over classes and take the average of outputs over images
5. Finally, take the exponential of the averaged value.


  


In [1]:
# calculate inception score in numpy
from numpy import asarray
from numpy import expand_dims
from numpy import log
from numpy import mean
from numpy import exp


In [2]:
# calculate the inception score for p(y|x)
def calculate_inception_score(p_yx, eps=1E-16):
 # calculate p(y)
 p_y = expand_dims(p_yx.mean(axis=0), 0)
 # kl divergence for each image
 kl_d = p_yx * (log(p_yx + eps) - log(p_y + eps))
 # sum over classes
 sum_kl_d = kl_d.sum(axis=1)
 # average over images
 avg_kl_d = mean(sum_kl_d)
 # undo the logs
 is_score = exp(avg_kl_d)
 return is_score


In [4]:
# conditional probabilities for low quality images
p_yx = asarray([[1.0, 0.0, 0.0], [0.0, 0.5, 0.5], [0.0, 0.5, 0.5]])
score = calculate_inception_score(p_yx)
print(score)

1.8898815748423095


# Inception Score for Cifar-10

In [2]:
import tensorflow as tf

device_name = tf.test.gpu_device_name()
if device_name != '/device:GPU:0':
  raise SystemError('GPU device not found')
print('Found GPU at: {}'.format(device_name))

Found GPU at: /device:GPU:0


In [2]:
# calculate inception score for cifar-10 in Keras
from math import floor
from numpy import ones
from numpy import expand_dims
from numpy import log
from numpy import mean
from numpy import std
from numpy import exp
from numpy.random import shuffle
from keras.applications.inception_v3 import InceptionV3
from keras.applications.inception_v3 import preprocess_input
from keras.datasets import cifar10
from skimage.transform import resize
from numpy import asarray



In [3]:
# scale an array of images to a new size
def scale_images(images, new_shape):
	images_list = list()
	for image in images:
		# resize with nearest neighbor interpolation
		new_image = resize(image, new_shape, 0)
		# store
		images_list.append(new_image)
	return asarray(images_list)

In [4]:
# assumes images have any shape and pixels in [0,255]
def calculate_inception_score(images, n_split=10, eps=1E-16):
	# load inception v3 model
	model = InceptionV3()
	# enumerate splits of images/predictions
	scores = list()
	n_part = floor(images.shape[0] / n_split)
	for i in range(n_split):
		# retrieve images
		ix_start, ix_end = i * n_part, (i+1) * n_part
		subset = images[ix_start:ix_end]
		# convert from uint8 to float32
		subset = subset.astype('float32')
		# scale images to the required size
		subset = scale_images(subset, (299,299,3))
		# pre-process images, scale to [-1,1]
		subset = preprocess_input(subset)
		# predict p(y|x)
		p_yx = model.predict(subset)
		# calculate p(y)
		p_y = expand_dims(p_yx.mean(axis=0), 0)
		# calculate KL divergence using log probabilities
		kl_d = p_yx * (log(p_yx + eps) - log(p_y + eps))
		# sum over classes
		sum_kl_d = kl_d.sum(axis=1)
		# average over images
		avg_kl_d = mean(sum_kl_d)
		# undo the log
		is_score = exp(avg_kl_d)
		# store
		scores.append(is_score)
	# average across images
	is_avg, is_std = mean(scores), std(scores)
	return is_avg, is_std


In [6]:
# load cifar10 images
(_, _), (images, _) = cifar10.load_data()
# shuffle images
shuffle(images)
print('loaded', images.shape)
# calculate inception score
is_avg, is_std = calculate_inception_score(images[:1000])
print('score', is_avg, is_std)

loaded (10000, 32, 32, 3)
4/4 [==============================] - 0s 101ms/step
score 9.278601 0.85542053
